> **Portfolio version.** Cell outputs and workspace-specific connection details have been removed. Configure the environment variables documented in the repository README before running on Databricks.


# Gold ? Borough-Year Analytical Panel

Joins the Silver domain tables and creates the final analysis-ready Gold table.

Crime, fly-tipping, population, income, and unemployment are annual borough-year features. CSI and IMD are static borough-level context features joined by borough only.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


## 1. Configuration

In [ ]:
catalog = ""
silver_schema = "cbda_silver"
gold_schema = "cbda_gold"
write_mode = "overwrite"
run_optimize = True

if catalog:
    silver_ns = f"`{catalog}`.`{silver_schema}`"
    gold_ns = f"`{catalog}`.`{gold_schema}`"
else:
    silver_ns = f"`{silver_schema}`"
    gold_ns = f"`{gold_schema}`"

print(f"Silver namespace: {silver_ns}")
print(f"Gold namespace: {gold_ns}")
print(f"Write mode: {write_mode}")


In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_ns}")


## 2. Helper Functions

In [ ]:
def table_name(namespace: str, table: str) -> str:
    return f"{namespace}.`{table}`"


def safe_rate(numerator_col: str, denominator_col: str, multiplier: float = 1.0):
    return (
        F.when(F.col(denominator_col).isNull() | (F.col(denominator_col) == 0), None)
        .otherwise((F.col(numerator_col) / F.col(denominator_col)) * F.lit(multiplier))
    )


def read_silver(table: str):
    return spark.table(table_name(silver_ns, table))


## 3. Read Silver Tables

In [ ]:
crime = (
    read_silver("silver_crime_borough_year")
    .select(
        "borough_key",
        F.col("borough_name").alias("crime_borough_name"),
        "financial_year",
        "fy_start_year",
        "crime_count",
        "crime_month_category_records",
        "major_crime_categories",
        "minor_crime_categories",
    )
)

fly = (
    read_silver("silver_flytipping_borough_year")
    .select(
        "borough_key",
        F.col("borough_name").alias("fly_borough_name"),
        "financial_year",
        "flytipping_incidents",
        "flytipping_actions",
        "flytipping_enforcement_rate",
    )
)

population = (
    read_silver("silver_population_borough_year")
    .select(
        "borough_key",
        F.col("borough_name").alias("population_borough_name"),
        "financial_year",
        "population_mid_year",
    )
)

income = (
    read_silver("silver_income_borough_year")
    .select(
        "borough_key",
        F.col("borough_name").alias("income_borough_name"),
        "financial_year",
        "income_taxpayer_count",
        "mean_income_gbp",
        "median_income_gbp",
    )
)

unemployment = (
    read_silver("silver_unemployment_borough_year")
    .select(
        "borough_key",
        F.col("borough_name").alias("unemployment_borough_name"),
        "financial_year",
        "unemployment_number_est_raw",
        "unemployment_denominator_est_raw",
        "unemployment_rate_est_raw",
        "unemployment_rate_est",
        "unemployment_rate_was_missing_raw",
        "unemployment_rate_was_imputed",
    )
)

csi = (
    read_silver("silver_csi_by_borough")
    .select(
        "borough_key",
        F.col("borough_name").alias("csi_borough_name"),
        "csi_overall_score",
        "csi_overall_score_unweighted",
        "csi_ward_count",
        "csi_population_sum",
    )
)

imd = read_silver("silver_imd_domains_by_borough").drop("borough_name")


## 4. Join Borough-Year 

In [ ]:
join_keys = ["borough_key", "financial_year"]

panel = (
    crime
    .join(fly, on=join_keys, how="left")
    .join(population, on=join_keys, how="left")
    .join(income, on=join_keys, how="left")
    .join(unemployment, on=join_keys, how="left")
    .join(csi, on="borough_key", how="left")
    .join(imd, on="borough_key", how="left")
    .withColumn(
        "borough_name",
        F.coalesce(
            "crime_borough_name",
            "fly_borough_name",
            "population_borough_name",
            "income_borough_name",
            "unemployment_borough_name",
            "csi_borough_name",
        ),
    )
    .where(F.col("borough_key").isNotNull())
    .where(F.col("financial_year").isNotNull())
    .withColumn("crime_rate_per_1000", F.round(safe_rate("crime_count", "population_mid_year", 1000.0), 3))
    .withColumn("flytipping_rate_per_1000", F.round(safe_rate("flytipping_incidents", "population_mid_year", 1000.0), 3))
    .withColumn("enforcement_rate", F.round(safe_rate("flytipping_actions", "flytipping_incidents", 1.0), 4))
    .withColumn("pre_covid_period", F.col("fy_start_year") <= 2018)
    .withColumn("covid_period", F.col("financial_year").isin("2019-20", "2020-21"))
    .withColumn("post_covid_period", F.col("fy_start_year") >= 2021)
)

drop_name_cols = [
    "crime_borough_name",
    "fly_borough_name",
    "population_borough_name",
    "income_borough_name",
    "unemployment_borough_name",
    "csi_borough_name",
]

for col_name in drop_name_cols:
    if col_name in panel.columns:
        panel = panel.drop(col_name)

front_cols = [
    "borough_key",
    "borough_name",
    "financial_year",
    "fy_start_year",
    "crime_count",
    "crime_rate_per_1000",
    "flytipping_incidents",
    "flytipping_rate_per_1000",
    "enforcement_rate",
    "population_mid_year",
    "median_income_gbp",
    "mean_income_gbp",
    "unemployment_rate_est",
    "unemployment_rate_was_imputed",
    "csi_overall_score",
    "living_environment_average_score",
    "income_average_score",
    "employment_average_score",
    "pre_covid_period",
    "covid_period",
    "post_covid_period",
]

front_cols = [c for c in front_cols if c in panel.columns]
remaining_cols = [c for c in panel.columns if c not in front_cols]
gold = panel.select(*front_cols, *remaining_cols).orderBy("borough_name", "fy_start_year")

display(gold)

## 5. Gold Quality Checks

In [ ]:
row_count = gold.count()
borough_count = gold.select("borough_key").distinct().count()
year_count = gold.select("financial_year").distinct().count()
duplicate_count = (
    gold.groupBy("borough_key", "financial_year")
    .count()
    .where(F.col("count") > 1)
    .count()
)

expected_rows = 32 * 13

print(f"Gold rows: {row_count}")
print(f"Boroughs: {borough_count}")
print(f"Financial years: {year_count}")
print(f"Duplicate borough-year keys: {duplicate_count}")
print(f"Expected borough-year rows: {expected_rows}")

if row_count != expected_rows:
    print("WARNING: Gold row count differs from expected 416. Check upstream missing keys or filters.")
if duplicate_count != 0:
    raise ValueError("Gold table contains duplicate borough-year keys.")

gold.select(
    F.count(F.lit(1)).alias("row_count"),
    F.countDistinct("borough_key").alias("borough_count"),
    F.countDistinct("financial_year").alias("financial_year_count"),
    F.sum(F.when(F.col("crime_rate_per_1000").isNull(), 1).otherwise(0)).alias("null_crime_rate_rows"),
    F.sum(F.when(F.col("flytipping_rate_per_1000").isNull(), 1).otherwise(0)).alias("null_flytipping_rate_rows"),
    F.sum(F.when(F.col("population_mid_year").isNull(), 1).otherwise(0)).alias("null_population_rows"),
    F.sum(F.when(F.col("median_income_gbp").isNull(), 1).otherwise(0)).alias("null_income_rows"),
    F.sum(F.when(F.col("unemployment_rate_est").isNull(), 1).otherwise(0)).alias("null_unemployment_rows"),
    F.sum(F.when(F.col("csi_overall_score").isNull(), 1).otherwise(0)).alias("null_csi_rows"),
).display()


## 6. Gold Delta Table

In [ ]:
gold_table = table_name(gold_ns, "gold_borough_year_panel")

(
    gold.write
    .format("delta")
    .mode(write_mode)
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

print(f"Wrote {gold_table}: rows={gold.count()}, cols={len(gold.columns)}")


In [ ]:
if run_optimize:
    try:
        spark.sql(f"OPTIMIZE {gold_table} ZORDER BY (borough_key, financial_year)")
        print("OPTIMIZE ZORDER completed.")
    except Exception as exc:
        print(f"OPTIMIZE skipped: {exc}")


In [ ]:
spark.sql(f"""
SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT borough_key) AS borough_count,
  COUNT(DISTINCT financial_year) AS financial_year_count,
  MIN(financial_year) AS first_financial_year,
  MAX(financial_year) AS last_financial_year,
  SUM(crime_count) AS total_crime_count,
  SUM(flytipping_incidents) AS total_flytipping_incidents
FROM {gold_table}
""").display()

spark.sql(f"""
SELECT borough_name, financial_year, crime_rate_per_1000, flytipping_rate_per_1000,
       median_income_gbp, unemployment_rate_est, csi_overall_score
FROM {gold_table}
ORDER BY borough_name, fy_start_year
LIMIT 20
""").display()


In [ ]:
# Optional Gold export to ADLS Gen2
import os

STORAGE_ACCOUNT = os.environ.get("AZURE_STORAGE_ACCOUNT")
GOLD_CONTAINER = os.environ.get("AZURE_GOLD_CONTAINER", "gold")
GOLD_PREFIX = os.environ.get("AZURE_GOLD_PREFIX", "")

if not STORAGE_ACCOUNT:
    raise EnvironmentError("Set AZURE_STORAGE_ACCOUNT before exporting Gold tables.")

gold_adls_path = (
    f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{GOLD_PREFIX}"
    .rstrip("/") + "/"
)

def anon(text):
    return str(text).replace(STORAGE_ACCOUNT, "***")

gold_tables = [
    row.tableName
    for row in spark.sql(f"SHOW TABLES IN {gold_ns}").collect()
    if not row.tableName.startswith("_")
]

print(f"Exporting {len(gold_tables)} Gold tables to ADLS...")
for table in gold_tables:
    df = spark.table(table_name(gold_ns, table))
    output = f"{gold_adls_path}{table}"
    df.write.format("delta").mode("overwrite").save(output)
    print(f"  {table} -> {anon(output)}")

print(f"All Gold tables exported to {anon(gold_adls_path)}")
